# Koddan Kasa · Levha 24 · İki doğru, bir yanlış

Videodaki deney: kuşbakışı 2D bir düzlemde robot (bir nokta) aşağıdan yukarıya gidiyor, arada yuvarlak bir engel var.
**200 gösterimin yarısı engelin solundan, yarısı sağından** geçiyor. İkisi de doğru.

- **MSE modeli:** başlangıç noktasına bakıp yolun tamamını (40 ara nokta) tahmin eden küçük bir MLP, kayıp = karesel hata.
- **Karışım modeli (MDN, 2 bileşen):** aynı ağ, ama çıkışı iki olası yol + ağırlıkları. Denemede önce bir bileşen seçilir (sol mu sağ mı), sonra o yol izlenir.

Videodaki her sayı bu defterin yazdığı `results/24/result.json` dosyasından gelir. Yalnız NumPy; GPU gerekmez (Colab CPU'da ~1 dk).

## 1 · Kurulum ve sürümler

In [ ]:
import json, os, sys, time
import numpy as np
print("python", sys.version.split()[0], "· numpy", np.__version__)
try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None
    print("matplotlib yok: çizim hücresi atlanacak")

## 2 · Gösterimler, ağ, deneme (videodaki deneyin aynısı: `src/v6/kk24/experiment.py`)

In [ ]:
START, GOAL = np.array([0.0, -1.0]), np.array([0.0, 1.0])
OBS_C, OBS_R = np.array([0.0, 0.0]), 0.30
T_DEMO = 40          # yol başına ara nokta
N_ROLLOUT = 200
EPOCHS = 1500
HIDDEN = 64
LR = 3e-3
SEED = 0
N_MAIN = 200         # ana koşudaki gösterim sayısı
SIGMA = 0.2          # karışım bileşenlerinin sabit σ'sı (ölçekli birimde)
SCALE = 2.0          # yol koordinatları ~0,5; ağ ~1 ölçeğinde öğrensin


# ── gösterimler ────────────────────────────────────────────────────────────────
def demo(rng, side, x0):
    """Engelin `side` (−1 sol, +1 sağ) yanından geçen yumuşak yol; ara noktanın yanal uzaklığı biraz oynar."""
    a = 0.55 + rng.normal(0, 0.04)          # engelin yanından geçerken yanal uzaklık (engel yarıçapı 0,30)
    s = np.linspace(0, 1, T_DEMO + 1)
    y = -1 + 2 * s
    x = x0 * (1 - s) + side * a * np.sin(np.pi * s) ** 1.2
    return np.stack([x, y], 1)


def make_data(rng, n_demo, left_frac):
    n_left = int(round(n_demo * left_frac))
    sides = np.array([-1] * n_left + [1] * (n_demo - n_left))
    rng.shuffle(sides)
    x0s = rng.uniform(-0.05, 0.05, n_demo)
    paths = np.stack([demo(rng, s, x0) for s, x0 in zip(sides, x0s)])
    X = paths[:, 0, :]                                       # başlangıç noktası
    Y = (paths[:, 1:, :] - paths[:, :1, :]).reshape(n_demo, -1)  # başlangıca göre yol
    return paths, sides, X, Y


# ── küçük MLP (2 gizli katman, tanh), Adam, tam yığın ─────────────────────────────
class MLP:
    def __init__(self, rng, n_in, n_out):
        sz = [n_in, HIDDEN, HIDDEN, n_out]
        self.W = [rng.normal(0, 1 / np.sqrt(a), (a, b)) for a, b in zip(sz[:-1], sz[1:])]
        self.b = [np.zeros(b) for b in sz[1:]]
        self.m = [np.zeros_like(p) for p in self.W + self.b]
        self.v = [np.zeros_like(p) for p in self.W + self.b]
        self.k = 0

    def forward(self, X):
        self.h = [X]
        for i, (W, b) in enumerate(zip(self.W, self.b)):
            z = self.h[-1] @ W + b
            self.h.append(np.tanh(z) if i < len(self.W) - 1 else z)
        return self.h[-1]

    def backward(self, g):
        gW, gb = [], []
        for i in reversed(range(len(self.W))):
            gW.insert(0, self.h[i].T @ g)
            gb.insert(0, g.sum(0))
            if i:
                g = (g @ self.W[i].T) * (1 - self.h[i] ** 2)
        self.k += 1
        for j, (p, d) in enumerate(zip(self.W + self.b, gW + gb)):
            self.m[j] = 0.9 * self.m[j] + 0.1 * d
            self.v[j] = 0.999 * self.v[j] + 0.001 * d * d
            p -= LR * (self.m[j] / (1 - 0.9 ** self.k)) / (np.sqrt(self.v[j] / (1 - 0.999 ** self.k)) + 1e-8)


def inp(X):
    return np.stack([X[:, 0] * 20, X[:, 1]], 1)  # x0 ±0,05 → ±1


def train_mse(rng, X, Y, epochs=EPOCHS):
    net = MLP(rng, 2, Y.shape[1])
    I, T = inp(X), Y * SCALE
    for _ in range(epochs):
        P = net.forward(I)
        net.backward(2 * (P - T) / T.size)
    return net, float(((net.forward(I) - T) ** 2).mean() / SCALE ** 2)


K = 2


def mdn_split(out, D):
    logit = out[:, :K]
    mu = out[:, K:K + K * D].reshape(-1, K, D)
    return logit, mu, np.full((len(out), K), np.log(SIGMA))  # σ sabit (izotropik): bileşenler çökmesin


def train_mdn(rng, X, Y, epochs=EPOCHS):
    D = Y.shape[1]
    net = MLP(rng, 2, K + K * D)
    I, T = inp(X), Y * SCALE
    n = len(X)
    # k-means++ gibi başlatma: bileşen ortalamaları rastgele bir gösterim ve ona en uzak gösterim (etiket kullanılmaz)
    i = int(rng.integers(n))
    j = int(np.argmax(((T - T[i]) ** 2).sum(1)))
    net.b[-1][K:] = np.concatenate([T[i], T[j]])
    net.W[-1][:, K:] *= 0.1
    for _ in range(epochs):
        logit, mu, logs = mdn_split(net.forward(I), D)
        lw = logit - logit.max(1, keepdims=True)
        lw = lw - np.log(np.exp(lw).sum(1, keepdims=True))
        d2 = ((T[:, None, :] - mu) ** 2).sum(2)
        s2 = np.exp(2 * logs)
        lp = lw - d2 / (2 * s2) - D * logs
        r = np.exp(lp - lp.max(1, keepdims=True))
        r /= r.sum(1, keepdims=True)                       # sorumluluklar
        g_logit = (np.exp(lw) - r) / n
        g_mu = (-(r / s2)[:, :, None] * (T[:, None, :] - mu)) / n
        g = np.concatenate([g_logit, g_mu.reshape(n, -1)], 1)
        net.backward(g)
    return net


# ── deneme ve çarpma ölçüsü ────────────────────────────────────────────────────
def first_hit(path):
    """Yolun engele ilk girdiği nokta (yoksa None) ve oraya kadarki kısım."""
    for i in range(len(path) - 1):
        p, q = path[i], path[i + 1]
        d = q - p
        for s in np.linspace(0, 1, 51):
            if np.linalg.norm(p + s * d - OBS_C) < OBS_R:
                return np.vstack([path[: i + 1], p + max(0.0, s - 0.02) * d])
    return None


def side_of(path):
    mid = path[np.argmin(np.abs(path[:, 1]))]
    return -1 if mid[0] < 0 else 1


def rollout_path(x0, rel):
    p0 = np.array([x0, START[1]])
    return np.vstack([p0, p0 + rel.reshape(-1, 2)])


def evaluate(predict, rng, n=N_ROLLOUT):
    xs = rng.uniform(-0.05, 0.05, n)
    rows = []
    for x0 in xs:
        full = rollout_path(x0, predict(x0))
        hit = first_hit(full)
        rows.append(dict(x0=float(x0), full=full, hit=hit is not None, drawn=hit if hit is not None else full, side=side_of(full)))
    hits = sum(r["hit"] for r in rows)
    clear = [np.min(np.linalg.norm(r["full"] - OBS_C, axis=1)) - OBS_R for r in rows]
    return dict(n=n, hits=int(hits), hit_pct=round(100 * hits / n, 1),
                left=int(sum(r["side"] < 0 and not r["hit"] for r in rows)), right=int(sum(r["side"] > 0 and not r["hit"] for r in rows)),
                mean_min_clearance=round(float(np.mean(clear)), 3)), rows


def mse_predict(net):
    return lambda x0: net.forward(inp(np.array([[x0, START[1]]])))[0] / SCALE


def mdn_predict(net, rng, D):
    def f(x0):
        logit, mu, _ = mdn_split(net.forward(inp(np.array([[x0, START[1]]]))), D)
        w = np.exp(logit[0] - logit[0].max())
        k = int(rng.choice(K, p=w / w.sum()))                # önce sol mu sağ mı
        return mu[0, k] / SCALE                               # sonra o yol
    return f

## 3 · Ana koşu: 200 gösterim, yarısı soldan

In [ ]:
paths, sides, X, Y = make_data(np.random.default_rng(SEED), N_MAIN, 0.5)
mse, mse_loss = train_mse(np.random.default_rng(SEED + 1), X, Y)
ev_mse, rows_mse = evaluate(mse_predict(mse), np.random.default_rng(SEED + 2))
mdn = train_mdn(np.random.default_rng(SEED + 1), X, Y)
ev_mdn, rows_mdn = evaluate(mdn_predict(mdn, np.random.default_rng(SEED + 3), 2 * T_DEMO), np.random.default_rng(SEED + 2))
print(f"gösterim: {(sides < 0).sum()} sol, {(sides > 0).sum()} sağ")
print(f"MSE modeli     : {ev_mse['hits']}/{ev_mse['n']} denemede çarptı")
print(f"karışım modeli : {ev_mdn['hits']}/{ev_mdn['n']} denemede çarptı · {ev_mdn['left']} soldan, {ev_mdn['right']} sağdan")
mse_c = rollout_path(0.0, mse_predict(mse)(0.0))
print(f"engelin hizasında (y = 0): gösterimler x = {paths[sides < 0, 20, 0].mean():+.2f} / {paths[sides > 0, 20, 0].mean():+.2f}, MSE yolu x = {mse_c[20, 0]:+.3f} (engel yarıçapı {OBS_R})")

In [ ]:
if plt:
    fig, ax = plt.subplots(1, 2, figsize=(9, 6), sharey=True)
    for a, rows, title in [(ax[0], rows_mse, "MSE modeli"), (ax[1], rows_mdn, "karışım modeli")]:
        for p in paths[:60]:
            a.plot(p[:, 0], p[:, 1], color="0.75", lw=0.6)
        for r in rows[::10]:
            a.plot(r["drawn"][:, 0], r["drawn"][:, 1], color="#FF7A2E", lw=2)
        a.add_patch(plt.Circle(OBS_C, OBS_R, color="0.4"))
        a.set_aspect("equal"); a.set_title(title)
    plt.show()

## 4 · Referans: gösterimlerin hepsi soldan olsaydı

In [ ]:
_, _, XL, YL = make_data(np.random.default_rng(SEED), N_MAIN, 1.0)
mseL, _ = train_mse(np.random.default_rng(SEED + 1), XL, YL)
ev_L, _ = evaluate(mse_predict(mseL), np.random.default_rng(SEED + 2))
print(f"hepsi soldan · MSE modeli: {ev_L['hits']}/{ev_L['n']} çarptı")

## 5 · Tarama: tek bir sayıyı değiştir

Videonun sorusu: **gösterimlerin %80'i soldan olsa MSE modeli çarpar mıydı?** `LEFT_FRACS` listesini değiştirip kendin dene.
İkinci tarama gösterim sayısı: az gösterimde model gösterimleri ezberliyor (başlangıçtaki küçük farkı sola/sağa bağlıyor) ve çoğu kez kurtuluyor; veri arttıkça ortalamayı daha iyi öğreniyor, yani daha sık çarpıyor.

In [ ]:
LEFT_FRACS = [0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
sweep_left = []
for lf in LEFT_FRACS:
    _, _, Xf, Yf = make_data(np.random.default_rng(SEED), N_MAIN, lf)
    net, _ = train_mse(np.random.default_rng(SEED + 1), Xf, Yf)
    ev, _ = evaluate(mse_predict(net), np.random.default_rng(SEED + 2))
    sweep_left.append(dict(left_frac=lf, mse_hit_pct=ev["hit_pct"]))
    print(f"soldan oranı {lf:.0%}: MSE modeli %{ev['hit_pct']} çarptı")

sweep_n = []
for n in [10, 20, 50, 100, 200, 500]:
    _, _, Xn, Yn = make_data(np.random.default_rng(SEED), n, 0.5)
    net, _ = train_mse(np.random.default_rng(SEED + 1), Xn, Yn)
    ev, _ = evaluate(mse_predict(net), np.random.default_rng(SEED + 2))
    sweep_n.append(dict(n_demo=n, mse_hit_pct=ev["hit_pct"]))
    print(f"{n:3d} gösterim (yarı yarıya): MSE modeli %{ev['hit_pct']} çarptı")

## 6 · Sonuç dosyası

In [ ]:
os.makedirs("results/24", exist_ok=True)
out = dict(numpy=np.__version__, seed=SEED, epochs=EPOCHS, n_demo=N_MAIN, rollouts=N_ROLLOUT,
           mse=ev_mse, mdn=ev_mdn, all_left=ev_L, sweep_left_frac=sweep_left, sweep_n_demo=sweep_n)
json.dump(out, open("results/24/result.json", "w"), ensure_ascii=False, indent=1)
print("→ results/24/result.json")

## Dene
- `SIGMA`'yı büyüt (0.2 → 2): karışım modeli iki yolu ayırabiliyor mu?
- Engeli küçült (`OBS_R = 0.1`): ortalama yol artık engelin yanından geçer mi?
- Adım adım bir politika yaz ((x, y) → bir sonraki adım). Bu düzende neden çoğu zaman kurtuluyor? (İpucu: ortadaki ortalama akış kararsız; küçük bir fark onu bir yana iter.)

> Not: Diffusion Policy ve ACT gibi gerçek yöntemler bu oyuncak modelle aynı değil; ortak fikir, tek bir ortalama yerine birden çok olası hareketi temsil edebilmek.